# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [42]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-08','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** line item (one item in one order).

*Note that I changed order 3 to have a different vendor_id to match the instructions in the notebook.*

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [43]:
import pandas as pd
lines = pd.json_normalize(orders_json, record_path='lines', meta=['order','vendor_id']) # Flatten
assert len(lines) == 4
assert sum(lines['qty']) == 7

lines.head()

,item,qty,order,vendor_id
0,Cheeseburger,2,1,V-01
1,Soda,1,1,V-01
2,Foam Finger,1,2,V-10
3,Hot Dog,3,3,V-08


After printing out the first 4 rows, I can see that the json was flattened into a dataframe that is easier to work with.

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [44]:
vendors = pd.json_normalize(vendors_json) # Normalize the vendors json
joined = lines.merge(vendors, on='vendor_id', how='left', indicator=True) # Left join

unmatched = joined[joined['_merge'] == 'left_only'] # Determine unmatched vendor
print('vendor missing:', unmatched['vendor_id'].unique())
print('units attached:', unmatched['qty'].sum())
joined.head()

vendor missing: ['V-08']
units attached: 3


,item,qty,order,vendor_id,name,zone,_merge
0,Cheeseburger,2,1,V-01,Hoos Burgers,A,both
1,Soda,1,1,V-01,Hoos Burgers,A,both
2,Foam Finger,1,2,V-10,Cav Merch,A,both
3,Hot Dog,3,3,V-08,NaN,NaN,left_only


V-08 is the vendor that did not match. There are 3 units attached to that vendor. It is missing a name and zone.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [45]:
joined['name'] = joined['name'].fillna('Unknown') # Fill in unmatched vendor
joined['zone'] = joined['zone'].fillna('Unknown') # Fill in unmatched zone
by_vendor = joined.groupby('name')['qty'].sum().sort_values(ascending=False) # Group by vendor name and sum
print('quantity per vendor:', by_vendor)

quantity per vendor: name
Hoos Burgers    3
Unknown         3
Cav Merch       1
Name: qty, dtype: int64


I included the unmatched vendor, but filled in its name and zone to be Unknown. I did this because we only have one unmatched vendor, so all items can be attributed to that vendor. This preserves the 3 units attributable to the unknown vendor rather than dropping them in a `groupby`.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [46]:
by_zone = joined.groupby('zone')['qty'].sum().sort_values(ascending=False) # Groups by zone and sum
print('quantity per zone:', by_zone)

quantity per zone: zone
A          4
Unknown    3
Name: qty, dtype: int64


Zone A sold the most items with 4 items. 3 items aren't matched to a zone. These are from vendor_id V-08 because it is missing name and zone information. Since I filled in the nan zone as "Unknown", my zone totals did add up to 7. However, these 3 items with an unknown zone could belong to any zone, including Zone A.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [47]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

has_lines = [l for l in ragged_json if 'lines' in l] # Only select the order numbers with line items
ragged = pd.json_normalize(has_lines, record_path='lines', meta=['order','vendor_id']) # Flatten

print('missing qty:', ragged['qty'].isna().sum())
print('zero qty:', (ragged['qty'] == 0).sum())
ragged.head()

missing qty: 1
zero qty: 0


,item,qty,order,vendor_id
0,Soda,2.0,4,V-01
1,Fries,NaN,6,V-01


I left the missing qty as nan because it would be misleading if I filled it in as 0. It would also not allow me to sum quantities if I filled it in with a string such as 'Unknown.' I also dropped order 5 because it had no line items, so the only values that would be included in an order 5 row would be `order` and `vendor_id`. That is not super helpful.

### Q6 — Validate

In [48]:
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

*a) 'One row per line item' is the useful shape because you can't differentiate between quantities and items in an order that had multiple lines, such as order 1, if you used one row per order. This is because the whole `lines` list would be in one cell in the dataframe. Question 3, asking for total quantity by vendor name, becomes easy when you use one row per line item, but it would be hard with one row per order. It would also be hard to determine which item sold the largest quanitity of units.*

*b) The practical difference is that a missing quantity means that it is unknown (i.e. we don't actually know how many of the item were sold), but a quantity of 0 is a real measured value. If you were to do arithmetic such as the average quantity sold by item, the 0 would be included in the calculation whereas NaN would be skipped. This would make the calculated average lower than it actually is since the 0 quantity pulls it down.*